**Lab 15 - GStreamer and Web Interface for AI Models**

Compute:  **CPU**

Time needed: **30-35 minutes**

*Learning Objectives:*
  - Create and use of GStreamer based pipelines
  - Create web interfaces for AI models

## Part A :  GStreamer based pipelines

**Step 1:**

Install packages for GStreamer such as,
- plugins (all types - base, good, bad and ugly)
- GStreamer libraries

*The install may take a few minutes.*

In [ ]:
%%bash

sudo apt update && sudo apt install -qq -y \
  libgstreamer1.0-dev \
  libgstreamer-plugins-base1.0-dev \
  libgstreamer-plugins-bad1.0-dev \
  gstreamer1.0-plugins-base \
  gstreamer1.0-plugins-good \
  gstreamer1.0-plugins-bad \
  gstreamer1.0-plugins-ugly \
  gstreamer1.0-libav \
  gstreamer1.0-tools \
  gstreamer1.0-x \
  gstreamer1.0-alsa \
  gstreamer1.0-gl \
  gstreamer1.0-gtk3 \
  gstreamer1.0-qt5 \
  gstreamer1.0-pulseaudio

**Step 2:**

Install [NNStreamer](https://nnstreamer.ai/) - a Linux Foundation Project for creating neural network inference pipelines

In [ ]:
%%bash

sudo apt-add-repository ppa:nnstreamer/ppa
sudo apt-key adv --keyserver keyserver.ubuntu.com --recv-keys CADA0F77901522B3
sudo apt install -qq -y nnstreamer nnstreamer-example nnstreamer-tensorflow2-lite

**Step 3:**

Check for successfull installation of the nnstreamer plugin by using the ***gst-inspect*** command.

The nnstreamer plugin contains 23 elements listed below.

*The tensor_filter element is used for model inferencing.*

In [ ]:
!gst-inspect-1.0 nnstreamer

**Step 4:**

Check out the location and details of the ***nnstreamer-example*** package.

- How many examples relate of inference of object detection model?

In [ ]:
%%bash
dpkg -L nnstreamer-example

**Step 5:**

- Fetch models and sample images used in previous labs for experimentation

In [ ]:
!wget -qq https://edge-ai-doulos.s3.us-west-2.amazonaws.com/Edge-AI-Upload.zip
!unzip -qq Edge-AI-Upload.zip

**Step 6:**

Create a GStreamer pipeline which performs the following functionality,
- Read an image file
- Carry output preprocessing of image
- Use tensorflow-lite interpreter to execute MobileNet classification model on the image
- Save the results to a text file

You can either run this GStreamer pipeline in the Jupyter notebook cell below or on a terminal.


In [ ]:
 %%bash

 gst-launch-1.0 \
    filesrc location=/content/Images/dog.jpg ! decodebin ! imagefreeze num-buffers=1 ! \
    videoconvert ! video/x-raw,format=RGB,framerate=30/1 ! videoscale ! \
    video/x-raw,width=224,height=224 ! \
    tensor_converter ! \
    tensor_transform mode=arithmetic option="typecast:float32,add:-128.0" ! \
    tensor_transform mode=arithmetic option="div:128.0" ! \
    tensor_filter framework=tensorflow-lite model=/content/Model-Files/mobilenet_v1_1.0_224.tflite ! \
    tensor_decoder mode=image_labeling option1=/content/Model-Files/ImageNet-labels.txt  ! \
    filesink location=/content/classification_output.txt

**These are the stages of the pipeline.  Correlate them with the text based pipeline.**

```text

  [ Input File ]
        │
        ▼
 ┌──────────────┐
 │   filesrc    │  location=/content/Images/dog.jpg
 └──────┬───────┘
        │ (Encoded Bytes)
        ▼
 ┌──────────────┐
 │  decodebin   │  Auto-detects & decodes compressed image
 └──────┬───────┘
        │ (Decoded Image Frame)
        ▼
 ┌──────────────┐
 │ imagefreeze  │  num-buffers=1 (Converts frame to static stream)
 └──────┬───────┘
        │ (Video Stream)
        ▼
 ┌──────────────┐
 │ videoconvert │  Caps: video/x-raw, format=RGB, framerate=30/1
 └──────┬───────┘
        │ (RGB Video Frames)
        ▼
 ┌──────────────┐
 │  videoscale  │  Caps: video/x-raw, width=224, height=224
 └──────┬───────┘
        │ (224x224 RGB Image)
        ▼
 ┌──────────────────┐
 │ tensor_converter │  Converts raw video buffers into GstTensors (uint8)
 └──────┬───────────┘
        │ Tensor [224, 224, 3] (uint8)
        ▼
 ┌──────────────────┐
 │ tensor_transform │  mode=arithmetic, option="typecast:float32,add:-128.0"
 └──────┬───────────┘
        │ Tensor [224, 224, 3] (float32, values: [-128.0, 127.0])
        ▼
 ┌──────────────────┐
 │ tensor_transform │  mode=arithmetic, option="div:128.0"
 └──────┬───────────┘
        │ Tensor [224, 224, 3] (float32, normalized: [-1.0, 1.0])
        ▼
 ┌──────────────────┐
 │  tensor_filter   │  framework=tensorflow-lite
 └──────┬───────────┘  model=/content/Model-Files/mobilenet_v1_1.0_224.tflite
        │ Raw Inference Output [1000] (float32 logits/scores)
        ▼
 ┌──────────────────┐
 │  tensor_decoder  │  mode=image_labeling
 └──────┬───────────┘  option1=/content/Model-Files/ImageNet-labels.txt
        │ Formatted Label String (e.g., "Beagle")
        ▼
 ┌──────────────┐
 │   filesink   │  location=/content/classification_output.txt
 └──────────────┘

**Exercise:**

Identify the pipeline elements in the above example that are from NNStreamer.

##Part B : Web User Interface(UI) for AI Models

**Step 1:**

Understand the code which constructs a simple web app interface using Gradio to process and display images based on user selection:

- fn=apply_image_filter: Sets the Python callback function that runs whenever an input changes. The callback function has three image filters - Grayscale, Invert and Sepia. A Sepia filter is a digital photo effect that turns an image into warm shades of brown and tan to give it an old, vintage look.

- inputs:
  gr.Image(..., type="numpy"): Creates an image upload box and passes the uploaded image into apply_image_filter directly as a NumPy array (RGB matrix).

  gr.Dropdown(...): Provides a drop-down menu with three filter choices, defaulting to "Grayscale".

- outputs=gr.Image(...): Displays the processed NumPy array image returned by apply_image_filter.

- title: Sets the title header at the top of the web UI.

**Step 2:**

- Execute the code and see a public link generated. Click on this click to open the web UI as a new tab.
- Input saved images from file or from camera and then select one of the three filters.
- Click the Submit button to see the output of the image filter.

In [ ]:
import gradio as gr
import numpy as np

def apply_image_filter(image, filter_type):
    if image is None:
        return None

    img = image.copy()
    if filter_type == "Grayscale":
        gray = np.dot(img[..., :3], [0.2989, 0.5870, 0.1140])
        img = np.stack((gray,) * 3, axis=-1).astype(np.uint8)
    elif filter_type == "Invert":
        img = 255 - img
    elif filter_type == "Sepia":
        sepia_filter = np.array([
            [0.393, 0.769, 0.189],
            [0.349, 0.686, 0.168],
            [0.272, 0.534, 0.131]
        ])
        img = np.clip(img.dot(sepia_filter.T), 0, 255).astype(np.uint8)

    return img

demo = gr.Interface(
    fn=apply_image_filter,
    inputs=[
        gr.Image(label="Upload Image", type="numpy"),
        gr.Dropdown(["Grayscale", "Invert", "Sepia"], label="Filter", value="Grayscale")
    ],
    outputs=gr.Image(label="Processed Output"),
    title="Image Filter Test Engine"
)

demo.launch()

In [ ]:
!pip install -q ai-edge-litert

**Step 3:**

Let us now use Gradio for creating a Teachable Machine like web interface.

We will build a tabbed web interface using Gradio Blocks to run an SSD MobileNet object detection model powered by LiteRT (TensorFlow Lite). The steps for generating a LiteRT inference are similar to the previous lab on object detection.

The components of the web UI are as below:
- gr.Blocks & gr.Tabs: Sets up a custom layout titled "LiteRT Edge AI Dashboard" with markdown headers and a tabbed navigation interface.

- gr.Row & gr.Column: Arranges the user interface side-by-side:

- Input Column (Left): An image input (img_in) accepting uploads or live webcam feeds as NumPy arrays, a confidence threshold slider (conf_slider) ranging from 0.1 to 1.0 (defaulting to 0.5), and a primary trigger button (btn_ssd).

- Output Column (Right): An image output display (img_out) for rendered bounding boxes and a text box (txt_out) for raw detection data/logs.

- btn_ssd.click(...): Binds the "Detect Objects" button to the Python function run_litert_ssd_mobilenet, passing the uploaded image and confidence value as arguments, and routing the returned annotated image and text detection summaries to the output components.

**Step 4:**

Experiment with the model inferencing web UI either below or as a separate browser tab.


In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
import gradio as gr
import ai_edge_litert.interpreter as litert

# ==========================================
# 1. EXPORT & PREPARE LITERT (.tflite) MODELS
# ==========================================

SSD_TFLITE_PATH = "/content/Model-Files/ssd_mobilenet_v1.tflite"
LABELS_PATH = "/content/Model-Files/labelmap.txt"

# Load labels for SSD MobileNet
if os.path.exists(LABELS_PATH):
    with open(LABELS_PATH, "r") as f:
        SSD_CLASSES = [line.strip() for line in f.readlines()]
else:
    SSD_CLASSES = []

# ==========================================
# 2. LITERT INTERPRETER INITIALIZATION
# ==========================================

# Initialize SSD LiteRT Interpreter
ssd_interpreter = litert.Interpreter(model_path=SSD_TFLITE_PATH)
ssd_interpreter.allocate_tensors()
ssd_input_details = ssd_interpreter.get_input_details()
ssd_output_details = ssd_interpreter.get_output_details()


# Helper to identify tensor index by name or shape
def _get_output_tensor(interpreter, output_details, keyword, default_idx):
    for detail in output_details:
        if keyword in detail["name"].lower():
            return interpreter.get_tensor(detail["index"])[0]
    return interpreter.get_tensor(output_details[default_idx]["index"])[0]


# ==========================================
# 3. LITERT INFERENCE PIPELINES
# ==========================================


def _preprocess_image_for_ssd(image_rgb, target_height, target_width, input_dtype):
    """Preprocesses an RGB image matching the TFLite model's input requirements."""
    # Resize image to expected input shape
    resized_image = cv2.resize(image_rgb, (target_width, target_height))

    # Handle Quantized (uint8/int8) vs Float32 model preprocessing
    if input_dtype == np.uint8:
        input_data = resized_image.astype(np.uint8)
    elif input_dtype == np.int8:
        input_data = (resized_image.astype(np.float32) - 128.0).astype(np.int8)
    else:
        # Standard MobileNet SSD Float32 normalization: [-1, 1]
        input_data = (resized_image.astype(np.float32) - 127.5) / 127.5

    # Add batch dimension [1, H, W, C]
    return np.expand_dims(input_data, axis=0)


def run_litert_ssd_mobilenet(image, conf_threshold):
    """Inference wrapper for SSD MobileNet using LiteRT execution."""
    if image is None:
        return None, "No input provided."

    # Image is passed in RGB from Gradio
    orig_h, orig_w = image.shape[:2]

    # Get model input shape and data type
    input_shape = ssd_input_details[0]["shape"]  # [1, H, W, 3]
    target_height, target_width = input_shape[1], input_shape[2]
    input_dtype = ssd_input_details[0]["dtype"]

    # Preprocess RGB input frame
    input_data = _preprocess_image_for_ssd(
        image, target_height, target_width, input_dtype
    )

    # Run LiteRT Inference
    ssd_interpreter.set_tensor(ssd_input_details[0]["index"], input_data)
    ssd_interpreter.invoke()

    # Retrieve output tensors dynamically
    output_boxes = _get_output_tensor(
        ssd_interpreter, ssd_output_details, "box", 0
    )
    output_classes = _get_output_tensor(
        ssd_interpreter, ssd_output_details, "class", 1
    )
    output_scores = _get_output_tensor(
        ssd_interpreter, ssd_output_details, "score", 2
    )
    num_detections_raw = _get_output_tensor(
        ssd_interpreter, ssd_output_details, "num", 3
    )

    num_detections = int(
        num_detections_raw[0]
        if isinstance(num_detections_raw, np.ndarray)
        else num_detections_raw
    )

    # Prepare image for drawing (OpenCV uses BGR)
    annotated_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    counts = {}

    for i in range(num_detections):
        score = float(output_scores[i])
        if score >= conf_threshold:
            # Normalized box coordinates [ymin, xmin, ymax, xmax]
            ymin, xmin, ymax, xmax = output_boxes[i]

            # Convert to pixel space
            x1 = int(max(0, xmin * orig_w))
            y1 = int(max(0, ymin * orig_h))
            x2 = int(min(orig_w, xmax * orig_w))
            y2 = int(min(orig_h, ymax * orig_h))

            class_id = int(output_classes[i]) + 1

            label = "Unknown"

            if class_id < len(SSD_CLASSES):
                label = SSD_CLASSES[class_id]
            else:
                label = f"Class {class_id}"

            if label.lower() == "background":
                continue

            counts[label] = counts.get(label, 0) + 1

            # Render bounding box and label text
            cv2.rectangle(annotated_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
            caption = f"{label} {score:.2f}"
            cv2.putText(
                annotated_bgr,
                caption,
                (x1, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                2,
            )

    # Convert annotated image back to RGB for Gradio display
    annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

    summary = (
        "\n".join([f"• {cls}: {cnt}" for cls, cnt in counts.items()])
        if counts
        else "No objects detected."
    )
    return annotated_rgb, summary


# ==========================================
# 4. GRADIO INTERFACE
# ==========================================

with gr.Blocks(title="LiteRT Edge AI Dashboard") as demo:
    gr.Markdown("# ⚡ LiteRT Edge Computer Vision")
    gr.Markdown(
        "Low-latency object detection accelerated via LiteRT (.tflite execution)."
    )

    with gr.Tabs():
        with gr.Tab("SSD MobileNet Object Detection (LiteRT)"):
            with gr.Row():
                with gr.Column():
                    img_in = gr.Image(
                        sources=["upload", "webcam"],
                        type="numpy",
                        label="Input Image",
                    )
                    conf_slider = gr.Slider(
                        0.1,
                        1.0,
                        value=0.5,
                        step=0.05,
                        label="Confidence Threshold",
                    )
                    btn_ssd = gr.Button("Detect Objects", variant="primary")
                with gr.Column():
                    img_out = gr.Image(label="Annotated Output")
                    txt_out = gr.Textbox(label="Detections")

            btn_ssd.click(
                fn=run_litert_ssd_mobilenet,
                inputs=[img_in, conf_slider],
                outputs=[img_out, txt_out],
            )

if __name__ == "__main__":
    demo.launch()

**Discussion:**

- Can this approach be used to create a UI for model training
- How can we create a web UI for classifying sound using YAMNet model?